# DDL, DML, Constraints, Upserts and Transactions

Notebook for **Chapter 9** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 DDL: CREATE, ALTER, DROP, Data Types and Constraints

## 9.2 DML: INSERT, UPDATE, DELETE, Upserts, Views and Friends

## 9.3 Transactions and ACID

## 9.4 Isolation Levels, MVCC, Locks and Deadlocks

In [ ]:
a.execute("BEGIN IMMEDIATE")
a.execute("UPDATE accounts SET balance = 700 WHERE id = 1")
print("A sees :", a.execute("SELECT balance FROM accounts").fetchone())
print("B sees :", b.execute("SELECT balance FROM accounts").fetchone())
b.execute("BEGIN IMMEDIATE")      # B also wants to write

A sees : (700,)
B sees : (1000,)
B error: database is locked
after commit B sees: (700,)


In [ ]:
bal_a = a.execute("SELECT balance FROM accounts").fetchone()[0]   # 1000
bal_b = b.execute("SELECT balance FROM accounts").fetchone()[0]   # 1000
a.execute("UPDATE accounts SET balance = ?", (bal_a - 100,))      # writes 900
b.execute("UPDATE accounts SET balance = ?", (bal_b - 200,))      # writes 800

expected 700, got: 800


In [ ]:
a.execute("UPDATE accounts SET balance = balance - 100")
b.execute("UPDATE accounts SET balance = balance - 200")

expected 700, got: 700


## 9.5 Transaction Anomaly Lab: Seeing Each Problem Happen

In [ ]:
import sqlite3, tempfile
d = tempfile.mkdtemp(); p = d + "/bank.db"
setup = sqlite3.connect(p); setup.execute("PRAGMA journal_mode=WAL")
setup.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER)")
setup.execute("INSERT INTO accounts VALUES (1, 1000)"); setup.commit(); setup.close()
a = sqlite3.connect(p, isolation_level=None, timeout=0)
b = sqlite3.connect(p, isolation_level=None, timeout=0)
b.execute("BEGIN")
print("B first read :", b.execute("SELECT balance FROM accounts WHERE id=1").fetchone()[0])
a.execute("BEGIN IMMEDIATE")
a.execute("UPDATE accounts SET balance = 900 WHERE id = 1")
a.execute("COMMIT")
print("B second read:", b.execute("SELECT balance FROM accounts WHERE id=1").fetchone()[0])
try:
    b.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
except sqlite3.OperationalError as e:
    print("B write error:", e)
b.execute("ROLLBACK")
print("B retry read :", b.execute("SELECT balance FROM accounts WHERE id=1").fetchone()[0])

B first read : 1000
B second read: 1000
B write error: database is locked
B retry read : 900


In [ ]:
import time, random
random.seed(1)
class Retryable(Exception):
    pass
def run_with_retry(job, tries=5):
    for attempt in range(1, tries + 1):
        try:
            return job(attempt)
        except Retryable as e:
            wait = 0.01 * 2 ** attempt + random.random() * 0.01
            print("attempt", attempt, "failed:", e, "| wait about", round(wait, 2), "s")
            time.sleep(wait)
    raise RuntimeError("gave up after %d tries" % tries)
def job(attempt):
    if attempt < 3:
        raise Retryable("deadlock detected")
    return "committed on attempt %d" % attempt
print(run_with_retry(job))

attempt 1 failed: deadlock detected | wait about 0.02 s
attempt 2 failed: deadlock detected | wait about 0.05 s
committed on attempt 3


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:", isolation_level=None)
db.execute("CREATE TABLE jobs (id INTEGER PRIMARY KEY, status TEXT, worker TEXT)")
db.executemany("INSERT INTO jobs (status) VALUES (?)", [("queued",)] * 3)
claim = """UPDATE jobs SET status = 'running', worker = ?
           WHERE id = (SELECT id FROM jobs WHERE status = 'queued' ORDER BY id LIMIT 1)
           RETURNING id"""
for w in ["w1", "w2", "w3", "w4"]:
    print(w, "claimed", db.execute(claim, (w,)).fetchall())
print(db.execute("SELECT * FROM jobs").fetchall())

w1 claimed [(1,)]
w2 claimed [(2,)]
w3 claimed [(3,)]
w4 claimed []
[(1, 'running', 'w1'), (2, 'running', 'w2'), (3, 'running', 'w3')]


## 9.6 Security: SQL Injection, Permissions and Row-Level Security

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, secret TEXT);
INSERT INTO customers VALUES (1,'Asha','Pune','card-1111'),(2,'Ravi','Mumbai','card-2222'),(3,'Meena','Pune','card-3333');
""")
def bad(city):
    sql = "SELECT name FROM customers WHERE city = '" + city + "'"
    return db.execute(sql).fetchall()
def good(city):
    return db.execute("SELECT name FROM customers WHERE city = ?", (city,)).fetchall()
evil = "x' UNION SELECT secret FROM customers --"
print("bad normal :", bad("Pune"))
print("bad evil   :", bad(evil))
print("good evil  :", good(evil))
try:
    bad("x'; DROP TABLE customers; --")
except Exception as e:
    print("stacked    :", type(e).__name__, e)

bad normal : [('Asha',), ('Meena',)]
bad evil   : [('card-1111',), ('card-2222',), ('card-3333',)]
good evil  : []
stacked    : ProgrammingError You can only execute one statement at a time.


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE products (name TEXT, price INTEGER)")
db.executemany("INSERT INTO products VALUES (?, ?)", [("Mug", 250), ("Pen Pack", 120), ("Lamp", 900)])
ALLOWED = {"name": "name", "price": "price"}
def list_products(sort_by):
    col = ALLOWED.get(sort_by)
    if col is None:
        raise ValueError("bad sort column: " + sort_by)
    return db.execute("SELECT name FROM products ORDER BY " + col).fetchall()
print(list_products("price"))
try:
    list_products("price; DROP TABLE products")
except ValueError as e:
    print("rejected:", e)

[('Pen Pack',), ('Mug',), ('Lamp',)]
rejected: bad sort column: price; DROP TABLE products


## 9.7 Programmability: Views, Triggers, Generated Columns, Identity and MERGE

## Chapter recap

## Practice set

Problem 9.P.1 (easy) Which constraint ensures `age` is between 0 and 120? **Answer:** `CHECK (age BETWEEN 0 AND 120)`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch09.html